# AUTOLOADER

In **Azure Databricks**, **Autoloader** is a powerful feature designed to incrementally and automatically ingest data from cloud storage into your tables. It simplifies the pipeline process by eliminating the need for manual file management and complex watermark logic.

**Key concepts of Autoloader:**
* **Incremental Loading:** It leverages *Spark Structured Streaming* to identify and process only new files in a source directory, ensuring efficiency by skipping previously processed data.
* **Idempotency:** It ensures an "exactly once" processing guarantee, meaning that even if the same file is re-scanned, the system avoids duplicate records in the destination.
* **Metadata Tracking:** It uses a hidden location (often referred to as *RocksDB*) to keep track of file states and metadata, effectively managing the ingestion status.
* **Schema Management:** Autoloader automatically detects and evolves schema changes. You can configure it to handle new columns by adding them, using a 'rescue' column for unexpected data, or failing the pipeline upon detection of significant changes.
* **Practical Usage:** It is commonly initialized using the `cloudFiles` format in Spark reads, and when combined with a `trigger(once=True)` setting, it is highly cost-effective for periodic batch-like ingestion.

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
        .option('cloudFiles.schemaLocation','/Volumes/real_time_catalog/source/real_time_volumne/checkPoint')\
                .load('/Volumes/real_time_catalog/source/real_time_volumne/')


In [0]:
df.writeStream.format('parquet')\
    .option('checkPointLocation','/Volumes/real_time_catalog/source/real_time_volumne/checkPoint')\
        .option('path','/Volumes/real_time_catalog/source/real_time_volumne/autoloader/')\
            .trigger(once=True)\
                .start()


In [0]:
%sql
Select * from parquet.`/Volumes/real_time_catalog/source/real_time_volumne/autoloader/`

In [0]:
# ============================================
# AUTO LOADER SCHEMA EVOLUTION MODES
# ============================================

# MODE 1: addNewColumns (DEFAULT)
# - Automatically adds new columns to the schema
# - Best for: Production pipelines where new columns are expected
df_add_new = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format', 'csv') \
    .option('cloudFiles.schemaLocation', '/Volumes/real_time_catalog/source/real_time_volumne/schema_add') \
    .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') \
    .option('cloudFiles.inferColumnTypes', 'true') \
    .load('/Volumes/real_time_catalog/source/real_time_volumne/')

print("Mode 1: addNewColumns - New columns are automatically added to schema")

# MODE 2: rescue
# - Puts unexpected columns into a _rescued_data column (JSON)
# - Best for: When you want to capture unexpected data without breaking the pipeline
df_rescue = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format', 'csv') \
    .option('cloudFiles.schemaLocation', '/Volumes/real_time_catalog/source/real_time_volumne/schema_rescue') \
    .option('cloudFiles.schemaEvolutionMode', 'rescue') \
    .option('cloudFiles.inferColumnTypes', 'true') \
    .load('/Volumes/real_time_catalog/source/real_time_volumne/')

print("Mode 2: rescue - Unexpected columns go into _rescued_data column")

# MODE 3: failOnNewColumns
# - Fails the stream when new columns are detected
# - Best for: Strict schemas where changes require manual review
df_fail = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format', 'csv') \
    .option('cloudFiles.schemaLocation', '/Volumes/real_time_catalog/source/real_time_volumne/schema_fail') \
    .option('cloudFiles.schemaEvolutionMode', 'failOnNewColumns') \
    .option('cloudFiles.inferColumnTypes', 'true') \
    .load('/Volumes/real_time_catalog/source/real_time_volumne/')

print("Mode 3: failOnNewColumns - Stream fails if new columns are detected")

# MODE 4: none
# - Disables schema evolution completely
# - Best for: When schema is fixed and managed externally
df_none = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format', 'csv') \
    .option('cloudFiles.schemaLocation', '/Volumes/real_time_catalog/source/real_time_volumne/schema_none') \
    .option('cloudFiles.schemaEvolutionMode', 'none') \
    .option('cloudFiles.inferColumnTypes', 'true') \
    .load('/Volumes/real_time_catalog/source/real_time_volumne/')

print("Mode 4: none - Schema evolution disabled")

# ============================================
# COMPARING BEHAVIORS
# ============================================
print("\n📊 Schema Evolution Mode Comparison:")
print("-" * 60)
print("addNewColumns:     ✅ Adds new columns automatically")
print("rescue:            ✅ Captures unexpected data in _rescued_data")
print("failOnNewColumns:  ❌ Stops stream on schema change")
print("none:              ❌ Ignores schema evolution")